# Notes

1. review math / pandas primers
2. go/sol for docs
3. try at least 1 Google search before asking for help/hints
4. working in groups welcome
5. Skews, Volpath, and QuadVar have (impossible) bonus exercises if you complete early
6. Jupyterlab shortcuts, TOC
6. give feedback on courses!

# Imports

In [ ]:
import pandas as pd

import solpy as sol

# Skew Parameters 

In [ ]:
reference_price = 1850
yte = 1 / 12
base_vol = 0.14
vol_path = 0
sd_points = list(range(-5, 6))
addends = [0.05, 0.04, 0.035, 0.0275, 0.0075, -0.005, 0, 0.005, 0.0125, 0.0275, 0.035]

In [ ]:
ax = pd.Series(addends, index=sd_points).plot()
ax.set_xlabel('Sd Point')
ax.set_ylabel('Vol')
ax.set_title('Vol By Sd');

# Create Sol Parameters

In [ ]:
params = sol.VolBySDModelParameters()

# parameters set above
params.refPrice = reference_price
params.baseVol = base_vol
params.sdVol = base_vol
params.volPathSlope = vol_path

# skew model types
params.volPathType = sol.VolatilityPathType.VOLPATHTYPE_Linear
params.sdCalcType = sol.StandardDeviationCalcType.SDTYPE_DiffWithATMVol
params.volExtrapMethod = sol.VolExtrapolationMethod.VOLEXTRAPMETHOD_Parametric
params.volSkewDiffType = sol.VolSkewDifferentialType.SKEWTYPE_Absolute
params.interpolationMethod = sol.InterpolationType.INTERPOLATIONMETHOD_ClampedCubicSpline

# other paraams
params.bendLower = 0
params.bendUpper = 0
params.speedLower = 1
params.speedUpper = 1
params.leftSlope = 0
params.rightSlope = 0

vol_input_type = sol.VolInputType.VOLTYPE_Lognormal

In [ ]:
sol_skew = sol.createStandardDeviationVolSkew(
    None, yte, params, sd_points, addends, vol_input_type
)

In [ ]:
sol_skew.toDictionary()

In [ ]:
strikes = range(1200, 2500, 25)
vols = sol.getVols(sol_skew, reference_price, strikes, yte)

In [ ]:
ax = pd.Series(vols, index=strikes).plot()
ax.set_xlabel('Strike')
ax.set_ylabel('Vol')
ax.set_title('Vol By Strike');

# Get Greeks 

In [ ]:
american_params = sol.AmericanAnalyticsParams()
american_params.americanAnalyticsType = sol.AmericanAnalyticsType.E_JZ

strikes = range(1600, 2100, 25)

repeated_strikes = list(strikes) * 2
option_types = [sol.CallPut_CALL] * len(strikes) + [sol.CallPut_PUT] * len(strikes)

greeks = sol.americanOptionWithVolSkewVec(
    1850,
    list(strikes) * 2,
    yte,
    sol_skew,
    0,
    0,
    option_types,
    american_params
)

In [ ]:
greeks['strike'] = repeated_strikes
greeks['option_type'] = option_types
greeks = pd.DataFrame(greeks)
greeks['option_type'] = greeks.option_type.map({sol.CallPut_CALL: 'C', sol.CallPut_PUT: 'P'})

In [ ]:
greeks.head()

In [ ]:
ax = (
    greeks
    .set_index(['strike', 'option_type'])
    .unstack('option_type')
    .Price
    .plot()
)
ax.set_title('Call and Put Prices')
ax.set_ylabel('Option Price')
ax.set_xlabel('Strike');